# Single-head causal attention, by hand

This is my first pass at self-attention, written before any polished version exists in the repo. I want to see every step and every shape, so I print the shape after each operation. Nothing here is optimised. The aim is to understand it properly.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)  # same numbers every run, so I can compare outputs
# small on purpose so I can actually read the tensors when I print them
# every size is different, so a shape mistake cannot hide behind a matching number
B = 2   # batch
T = 5   # sequence length
C = 8   # embedding size
head_size = 3  # one head, so this is just how wide Q, K, V are

print(B, T, C, head_size)

## The input

`x` stands in for a batch of embedded tokens, with shape (batch, time, channels). I use random numbers for now, because the point is to follow the shapes, not to learn anything yet.

In [ ]:
x = torch.randn(B, T, C)
print(x.shape) # (B, T, C) = (2, 5, 8)

## Queries, keys and values

I pass the same `x` through three separate linear layers. The query is what a token is looking for, the key is what a token offers, and the value is what it hands over if it is chosen. I leave out the bias so the layers stay simple, and I project down to `head_size` so each head works in a smaller space.

In [ ]:
query = nn.Linear(C, head_size, bias=False)
key   = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

q = query(x)
k = key(x)
v = value(x)

print(q.shape)  # (B, T, head_size) = (2, 5, 3)
print(k.shape)  # same
print(v.shape)  # same

## Scores

Every query is compared with every key using a dot product, so a high score means a strong match. That gives one score for each pair of tokens, which is why the result is T by T for each item in the batch. I chose `T` and `head_size` to be different on purpose, so if I forget the transpose, the shapes fail loudly instead of quietly passing.

In [ ]:
scores = q @ k.transpose(-2, -1)
print(scores.shape)  # (B, T, T) = (2, 5, 5)  one score per token pair

## Scaling

I divide the scores by the square root of `head_size`. If q and k have independent components with unit variance, the variance of their dot product equals `head_size`, so it grows with the width of the head. Large scores push the softmax towards a single hard winner, which makes learning harder. Dividing by the square root of `head_size` divides the variance by exactly `head_size`, which brings it back to about 1.

My own scores start with a variance below 1, because the default initialisation of `nn.Linear` does not give unit-variance q and k. So I check the claim a second time with random vectors that really do have unit variance.

In [ ]:
scaled = scores / head_size**0.5
print(scores.var())
print(scaled.var())  # exactly head_size times smaller, by definition

# the real argument: unit-variance q and k give a dot product with variance of about their width
wide = 64
q_demo = torch.randn(1000, wide)
k_demo = torch.randn(1000, wide)
raw = q_demo @ k_demo.T
print(raw.var())                  # about 64
print((raw / wide**0.5).var())    # about 1

## The Causal Mask

A token must not look at the future, otherwise the model could simply copy the next word while training and learn nothing. I build a lower-triangular boolean mask once, at the longest length I expect (`block_size`), and slice out the T by T corner I need each time. Building it once avoids allocating a new one on every call. I set every position above the diagonal to minus infinity, and `.to(x.device)` keeps the mask on the same device as the scores, which matters on a GPU. After the softmax, those positions become exactly zero.

In [ ]:
block_size = 8  # longest sequence this mask can serve, T is sliced out of it
mask = torch.tril(torch.ones(block_size, block_size, dtype=torch.bool))  # True = allowed to look

masked = scaled.masked_fill(~mask[:T, :T].to(scaled.device), float("-inf"))

print(mask[:T, :T].int())
print(masked[0, 0])  # first token: only itself is finite

## Softmax

Softmax along the last dimension turns each row of scores into weights that add up to one. Row `i` now says how much token `i` draws from each token up to and including itself.

In [ ]:
weights = F.softmax(masked, dim=-1)

print(weights.sum(dim=-1))          # all 1s
print(weights[0].triu(diagonal=1))  # zeros above the diagonal

## Weighted sum

Last, I use the weights to take a weighted average of the values. Each token's output is a blend of the values from the tokens it is allowed to see, so it carries information from its context. The output shape matches the value shape, (batch, time, head_size).

In [ ]:
out = weights @ v
print(out.shape)  # (B, T, head_size) = (2, 5, 3)  same as v

## Proving the mask works

I do not want to trust the mask by eye, so I test it. For every position `t`, I replace that token with new random numbers. Every output before `t` must stay exactly the same, because the past cannot see the future, and the output at `t` itself must change. The second check matters, because a function that ignored its input would pass the first one without proving anything. I use `torch.equal` and not `torch.allclose`, since the masked weights are exactly zero, so the earlier rows should be identical bit for bit.

In [ ]:
def causal_attn(x):
    T = x.size(1)
    q = query(x)
    k = key(x)
    v = value(x)
    scores = q @ k.transpose(-2, -1) / head_size**0.5
    scores = scores.masked_fill(~mask[:T, :T].to(x.device), float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return weights @ v

out1 = causal_attn(x)

for t in range(T):
    x2 = x.clone()
    x2[:, t, :] = torch.randn(B, C)
    out2 = causal_attn(x2)

    assert torch.equal(out1[:, :t, :], out2[:, :t, :]), f"leak: changing token {t} moved an earlier output"
    assert not torch.allclose(out1[:, t, :], out2[:, t, :]), f"output {t} ignored its own input"

print("past outputs unchanged, and the changed position moved, for every t")

## Attention on its own cannot tell the order

There is no position information in `x`, so attention only sees a set of tokens. To check, I reverse the order of every token before the last one and leave the last token where it is. Its output should not change, because it sees the same tokens in a different order. The causal mask is the only order signal here, and all it tells a position is how many tokens it is allowed to see, not the order they came in. This is why positional embeddings and RoPE exist.

In [ ]:
x3 = x.clone()
x3[:, :-1, :] = x[:, :-1, :].flip(1)  # reverse everything before the last token
out3 = causal_attn(x3)

# the last token sees the same tokens, so its output matches (allclose, as the sum runs in a new order)
assert torch.allclose(out1[:, -1, :], out3[:, -1, :])
# the reordering was real, because the earlier outputs did change
assert not torch.allclose(out1[:, :-1, :], out3[:, :-1, :])
print("last output unchanged by reordering the earlier tokens")

## What this leaves out

This is one head with no learned position information, so as the last test shows, it cannot tell token order. Multi-head attention, positional encoding (including RoPE) and dropout come next, once I have properly read up on them. I will tidy this into a proper module in the model file, and keep this notebook as it is.